# Structural Coverage Exercise

In this exercise, we'll practice:
- **Instrumentation** for testing (tracking coverage)
- **Coverage calculation** (statement, branch, condition, MCDC, and path coverage)
- **Manual test suite generation** to achieve high coverage

This hands-on approach will help you deeply understand how structural coverages work.

## Software under Test (SuT)

We'll test a function `calculate_shipping_fee` that calculates shipping fees based on six inputs.

Inputs:
- `order_total: int`
- `weight_kg: float`
- `distance_km: int`
- `is_island: bool`
- `membership: str`
- `coupon_type: str`

The function returns an integer shipping fee based on:
- base fee rule (`order_total < 40000`)
- weight surcharge (`<=5`, `<=20`, `>20`)
- distance surcharge (`<=10`, `<=50`, `>50`)
- island surcharge
- WOW membership discount
- NEW_USER coupon discount
- final lower bound with `max(fee, 0)`


In [1]:
def calculate_shipping_fee(
    order_total: int,
    weight_kg: float,
    distance_km: int,
    is_island: bool,
    membership: str,
    coupon_type: str
) -> int:
    fee = 0

    # 1. base fee
    if order_total < 40000:
        fee += 3000

    # 2. weight surcharge
    if weight_kg <= 5:
        fee += 0
    elif weight_kg <= 20:
        fee += 2000
    else:
        fee += 5000

    # 3. distance surcharge
    if distance_km <= 10:
        fee += 0
    elif distance_km <= 50:
        fee += 1000
    else:
        fee += 3000

    # 4. island surcharge
    if is_island:
        fee += 4000

    # 5. membership discount
    if membership == "WOW":
        fee = fee // 2

    # 6. coupon discount
    if coupon_type == "NEW_USER":
        fee -= 2000

    # 7. final lower bound
    return max(fee, 0)


## Instrumented SuT

To calculate test coverage, the SuT should be **instrumented** or **observed** during execution.

Tools like `PyTest` support automatic coverage tracking by:
1. Parsing the source code (AST manipulation)
2. Inserting tracking statements at strategic points
3. Recording which code elements were executed
4. Generating coverage reports

However, the tools usually support only some of coverages. 
(e.g., PyTest only supports statement branch coverage.)

In this exercise, we'll implement manual instrumentation to:
- understand how coverage tracking works internally
- Track all types of coverage: statement, branch, condition, MCDC and path
- See exactly what data needs to be collected for each coverage type

### Implementation Approach

We'll implement a `CoverageTracker` base class that will be the parent class for each type of coverage tracker.

`CoverageTracker` Responsibilities:
- **Contains** the instrumented SuT (instrumented `calculate_shipping_fee`)
- **Resets** coverage data between test runs
- **Runs** the SuT and tracks coverage for each test input
- **Calculates** coverage metrics
- **Reports** coverage results

Each child class (`StatementCoverageTracker`, `BranchCoverageTracker`, etc.) will inherit from `CoverageTracker` and implement its specific tracking logic.

In [2]:
from abc import ABC, abstractmethod
from typing import List, Tuple
from tabulate import tabulate


class CoverageTracker(ABC):
    """
    Abstract base class for coverage tracking.

    Stores coverage as: [(input, coverage_list, result), ...]
    where coverage_list is [(item_id, bool), ...]
    """

    def __init__(self):
        """Initialize the coverage tracker."""
        self.coverage_items = []  # List of item IDs to track: [item_id, item_id, ...]
        self.executions = []  # List of (input, coverage, result) triples
        self._tracking = {}  # Temporary tracking dict for current test
        self._define_coverage_items()

    @abstractmethod
    def _define_coverage_items(self):
        """
        Define coverage items to track for this coverage type.
        Must be implemented by child classes.
        
        Example for statements:
            self.coverage_items = ["stm1", "stm2", "stm3", ...]
        """
        pass

    def reset(self):
        """Reset the coverage tracker."""
        self.executions = []
        self._tracking = {}
        self._define_coverage_items()

    def init_tracking(self):
        """Initialize tracking for a new test execution."""
        self._tracking = {item_id: False for item_id in self.coverage_items}

    def track(self, item_id: str):
        """Mark an item as covered in the current test execution."""
        if item_id in self._tracking:
            self._tracking[item_id] = True

    def get_tracking_result(self) -> List[Tuple[str, bool]]:
        """Get the coverage list for the current test execution."""
        return [(item_id, covered) for item_id, covered in self._tracking.items()]

    def run_test(
        self,
        order_total: int,
        weight_kg: float,
        distance_km: int,
        is_island: bool,
        membership: str,
        coupon_type: str,
    ) -> int:
        test_input = (order_total, weight_kg, distance_km, is_island, membership, coupon_type)
        coverage, result = self.instrumented_calculate_shipping_fee(
            order_total, weight_kg, distance_km, is_island, membership, coupon_type
        )
        self.executions.append((test_input, coverage, result))
        return result

    @abstractmethod
    def instrumented_calculate_shipping_fee(
        self,
        order_total: int,
        weight_kg: float,
        distance_km: int,
        is_island: bool,
        membership: str,
        coupon_type: str,
    ) -> Tuple[List[Tuple[str, bool]], int]:
        """
        Instrumented version of calculate_shipping_fee.
        """
        pass

    def _get_covered_items(self):
        """
        Get the set of items covered across all executions.
        
        Returns:
            set: Set of covered item IDs
        """
        covered_items = set()
        for _, coverage, _ in self.executions:
            for item_id, was_covered in coverage:
                if was_covered:
                    covered_items.add(item_id)
        return covered_items

    def calculate_coverage(self) -> Tuple[float, int, int]:
        """
        Calculate overall coverage from all executions.
        
        Returns:
            Tuple[float, int, int]: (coverage_percentage, covered_items, total_items)
        """
        if not self.executions:
            return 0.0, 0, len(self.coverage_items)

        covered_items = self._get_covered_items()
        total_items = len(self.coverage_items)
        covered_count = len(covered_items)
        percentage = (covered_count / total_items * 100) if total_items > 0 else 0.0

        return percentage, covered_count, total_items

    def print_report(self):
        """Generate and print coverage report in table format using tabulate."""
        if not self.executions:
            print("No test executions recorded.")
            return

        # Calculate overall coverage
        percentage, covered, total = self.calculate_coverage()
        covered_items = self._get_covered_items()

        # Get coverage type name from class name
        coverage_type = self.__class__.__name__.replace("CoverageTracker", "").replace("Tracker", "")

        print("=" * 120)
        print(f"{coverage_type.upper()} COVERAGE REPORT")
        print("=" * 120)
        print(f"\nOverall Coverage: {percentage:.2f}% ({covered}/{total} items)")
        print(f"Total Tests: {len(self.executions)}\n")

        # Build table data
        headers = ["Test input"] + [test_input for test_input, _, _ in self.executions] + ["Covered"]

        # Coverage rows - use boolean directly
        table_data = []
        for item_id in self.coverage_items:
            row = [item_id] + [
                "O" if dict(coverage).get(item_id, False) else "X"
                for _, coverage, _ in self.executions
            ] + ["O" if item_id in covered_items else "X"]
            table_data.append(row)
        table_data.append([])

        # Result row
        result_row = ["Result"] + [result for _, _, result in self.executions] + [""]
        table_data.append(result_row)

        # Print tables with double-line separator
        print(tabulate(table_data, headers=headers, tablefmt="github"))
        print("=" * 120)

        def get_coverage_percentage(self) -> float:
            """Get coverage percentage."""
            percentage, _, _ = self.calculate_coverage()
            return percentage
        
        def get_executions(self) -> List[Tuple[Tuple[int, float, int, bool, str, str], List[Tuple[str, bool]], int]]:
            """
            Get all execution records.
            
            Returns:
                List of (input, coverage, result) triples
            """
            return self.executions


## Statement Coverage

Let's implement `StatementCoverageTracker` and test the SuT `calculate_shipping_fee` guided by the statement coverage.

### Statement Coverage Tracker Implementation

Statement coverage counts executed statements by tests over total statements in the SuT.

**TODO**: Implement a class `StatementCoverageTracker` overiding abstract classes:
- `_define_coverage_items(self)`: define an attribute `self.coverage_items`
- `instrumented_calculate_shipping_fee(self, ...)`: re-write `calculate_shipping_fee(...)` with `self.init_track`, `self.track`, and `self.get_tracking_result`.

In [3]:
class StatementCoverageTracker(CoverageTracker):
    """Tracks statement coverage for calculate_shipping_fee."""

    def _define_coverage_items(self):
        self.coverage_items = [
            "fee = 0",
            "fee += 3000",
            "fee += 0 (weight)",
            "fee += 2000",
            "fee += 5000",
            "fee += 0 (distance)",
            "fee += 1000",
            "fee += 3000 (distance)",
            "fee += 4000",
            "fee = fee // 2",
            "fee -= 2000",
            "return max(fee, 0)",
        ]

    def instrumented_calculate_shipping_fee(self, order_total, weight_kg, distance_km, is_island, membership, coupon_type):
        self.init_tracking()
        self.track("fee = 0")
        fee = 0

        if order_total < 40000:
            self.track("fee += 3000")
            fee += 3000

        if weight_kg <= 5:
            self.track("fee += 0 (weight)")
            fee += 0
        elif weight_kg <= 20:
            self.track("fee += 2000")
            fee += 2000
        else:
            self.track("fee += 5000")
            fee += 5000

        if distance_km <= 10:
            self.track("fee += 0 (distance)")
            fee += 0
        elif distance_km <= 50:
            self.track("fee += 1000")
            fee += 1000
        else:
            self.track("fee += 3000 (distance)")
            fee += 3000

        if is_island:
            self.track("fee += 4000")
            fee += 4000

        if membership == "WOW":
            self.track("fee = fee // 2")
            fee = fee // 2

        if coupon_type == "NEW_USER":
            self.track("fee -= 2000")
            fee -= 2000

        self.track("return max(fee, 0)")
        return self.get_tracking_result(), max(fee, 0)


### Test with Statement Coverage

**TODO**: Manually generate test inputs and run test:
- Find test inputs to improve statement coverage.
- Run tests and check the coverage report.

In [7]:
# Create tracker instance
stmt_tracker = StatementCoverageTracker()

#TODO: Generate test inputs and run tests to improve statement coverage
# Run multiple tests
stmt_tracker.run_test(10000, 3.0, 5, False, "NONE", "NONE")
stmt_tracker.run_test(10000, 3.0, 5, False, "NONE", "NEW_USER")

print()
stmt_tracker.print_report()



STATEMENT COVERAGE REPORT

Overall Coverage: 50.00% (6/12 items)
Total Tests: 2

| Test input             | (10000, 3.0, 5, False, 'NONE', 'NONE')   | (10000, 3.0, 5, False, 'NONE', 'NEW_USER')   | Covered   |
|------------------------|------------------------------------------|----------------------------------------------|-----------|
| fee = 0                | O                                        | O                                            | O         |
| fee += 3000            | O                                        | O                                            | O         |
| fee += 0 (weight)      | O                                        | O                                            | O         |
| fee += 2000            | X                                        | X                                            | X         |
| fee += 5000            | X                                        | X                                            | X         |
| fee += 0 (dis

## Branch (Decision) Coverage

Let's implement `BranchCoverageTracker` and test the SuT `calculate_shipping_fee` guided by the branch coverage.

### Branch Coverage Tracker Implementation

Branch coverage counts the execution of each possible branch (True/False outcome) of decision points in the system under test (SuT).

**TODO**: Implement a class `BranchCoverageTracker` overiding abstract classes::
- `_define_coverage_items(self)`: define an attribute `self.coverage_items`
- `instrumented_calculate_shipping_fee(self, ...)`: re-write `calculate_shipping_fee(...)` with `self.init_track`, `self.track`, and `self.get_tracking_result`.

In [8]:
class BranchCoverageTracker(CoverageTracker):
    """Tracks branch coverage for calculate_shipping_fee."""

    def _define_coverage_items(self):
        self.coverage_items = [
            "order_total < 40000: True", "order_total < 40000: False",
            "weight_kg <= 5: True", "weight_kg <= 5: False",
            "weight_kg <= 20: True", "weight_kg <= 20: False",
            "distance_km <= 10: True", "distance_km <= 10: False",
            "distance_km <= 50: True", "distance_km <= 50: False",
            "is_island: True", "is_island: False",
            "membership == WOW: True", "membership == WOW: False",
            "coupon_type == NEW_USER: True", "coupon_type == NEW_USER: False",
        ]

    def instrumented_calculate_shipping_fee(self, order_total, weight_kg, distance_km, is_island, membership, coupon_type):
        self.init_tracking()
        fee = 0

        self.track(f"order_total < 40000: {order_total < 40000}")
        if order_total < 40000:
            fee += 3000

        self.track(f"weight_kg <= 5: {weight_kg <= 5}")
        if weight_kg <= 5:
            pass
        else:
            self.track(f"weight_kg <= 20: {weight_kg <= 20}")
            if weight_kg <= 20:
                fee += 2000
            else:
                fee += 5000

        self.track(f"distance_km <= 10: {distance_km <= 10}")
        if distance_km <= 10:
            pass
        else:
            self.track(f"distance_km <= 50: {distance_km <= 50}")
            if distance_km <= 50:
                fee += 1000
            else:
                fee += 3000

        self.track(f"is_island: {is_island}")
        if is_island:
            fee += 4000

        wow = membership == "WOW"
        self.track(f"membership == WOW: {wow}")
        if wow:
            fee = fee // 2

        new_user = coupon_type == "NEW_USER"
        self.track(f"coupon_type == NEW_USER: {new_user}")
        if new_user:
            fee -= 2000

        return self.get_tracking_result(), max(fee, 0)


### Test with Branch Coverage

**TODO**: Manually generate test inputs and run test:
- Find test inputs to improve branch coverage.
- Run tests and check the coverage report.

In [10]:
# Create tracker instance
branch_tracker = BranchCoverageTracker()

branch_tracker.run_test(10000, 3.0, 5, False, "NONE", "NONE")
branch_tracker.run_test(10000, 3.0, 5, False, "NONE", "NEW_USER")

print()
branch_tracker.print_report()



BRANCH COVERAGE REPORT

Overall Coverage: 43.75% (7/16 items)
Total Tests: 2

| Test input                     | (10000, 3.0, 5, False, 'NONE', 'NONE')   | (10000, 3.0, 5, False, 'NONE', 'NEW_USER')   | Covered   |
|--------------------------------|------------------------------------------|----------------------------------------------|-----------|
| order_total < 40000: True      | O                                        | O                                            | O         |
| order_total < 40000: False     | X                                        | X                                            | X         |
| weight_kg <= 5: True           | O                                        | O                                            | O         |
| weight_kg <= 5: False          | X                                        | X                                            | X         |
| weight_kg <= 20: True          | X                                        | X                    

## Path Coverage

Let's implement `PathCoverageTracker` and test the SuT `calculate_shipping_fee` guided by the path coverage.

### Path Coverage Tracker Implementation

Path coverage counts the executed paths by tests over all paths in the system under test (SuT).

**TODO**: Implement a class `PathCoverageTracker` overiding abstract classes::
- `_define_coverage_items(self)`: define an attribute `self.coverage_items`
- `instrumented_calculate_shipping_fee(self, ...)`: re-write `calculate_shipping_fee(...)` with `self.init_track`, `self.track`, and `self.get_tracking_result`.

In [11]:
class PathCoverageTracker(CoverageTracker):
    """Tracks path coverage with branch-selection signature strings."""

    def _define_coverage_items(self):
        # Branch ids
        # 1: order_total < 40000 -> {0 if, 1 else}
        # 2: weight_kg <= 5 / <= 20 / else -> {0, 1, 2}
        # 3: distance_km <= 10 / <= 50 / else -> {0, 1, 2}
        # 4: is_island -> {0 if, 1 else}
        # 5: membership == WOW -> {0 if, 1 else}
        # 6: coupon_type == NEW_USER -> {0 if, 1 else}
        self.coverage_items = [
            f"1-{b1},2-{b2},3-{b3},4-{b4},5-{b5},6-{b6}"
            for b1 in (0, 1)
            for b2 in (0, 1, 2)
            for b3 in (0, 1, 2)
            for b4 in (0, 1)
            for b5 in (0, 1)
            for b6 in (0, 1)
        ]

    def instrumented_calculate_shipping_fee(self, order_total, weight_kg, distance_km, is_island, membership, coupon_type):
        self.init_tracking()
        fee = 0

        # Branch 1: base fee
        if order_total < 40000:
            b1 = 0
            fee += 3000
        else:
            b1 = 1

        # Branch 2: weight surcharge
        if weight_kg <= 5:
            b2 = 0
            fee += 0
        elif weight_kg <= 20:
            b2 = 1
            fee += 2000
        else:
            b2 = 2
            fee += 5000

        # Branch 3: distance surcharge
        if distance_km <= 10:
            b3 = 0
            fee += 0
        elif distance_km <= 50:
            b3 = 1
            fee += 1000
        else:
            b3 = 2
            fee += 3000

        # Branch 4: island surcharge
        if is_island:
            b4 = 0
            fee += 4000
        else:
            b4 = 1

        # Branch 5: membership discount
        if membership == "WOW":
            b5 = 0
            fee = fee // 2
        else:
            b5 = 1

        # Branch 6: coupon discount
        if coupon_type == "NEW_USER":
            b6 = 0
            fee -= 2000
        else:
            b6 = 1

        path_name = f"1-{b1},2-{b2},3-{b3},4-{b4},5-{b5},6-{b6}"
        self.track(path_name)

        return self.get_tracking_result(), max(fee, 0)


### Test with Path Coverage

**TODO**: Manually generate test inputs and run test:
- Find test inputs to improve path coverage.
- Run tests and check the coverage report.

In [13]:
# Create tracker instance
path_tracker = PathCoverageTracker()

path_tracker.run_test(10000, 3.0, 5, False, "NONE", "NONE")
path_tracker.run_test(10000, 3.0, 5, False, "NONE", "NEW_USER")

print()
path_tracker.print_report()



PATH COVERAGE REPORT

Overall Coverage: 1.39% (2/144 items)
Total Tests: 2

| Test input              | (10000, 3.0, 5, False, 'NONE', 'NONE')   | (10000, 3.0, 5, False, 'NONE', 'NEW_USER')   | Covered   |
|-------------------------|------------------------------------------|----------------------------------------------|-----------|
| 1-0,2-0,3-0,4-0,5-0,6-0 | X                                        | X                                            | X         |
| 1-0,2-0,3-0,4-0,5-0,6-1 | X                                        | X                                            | X         |
| 1-0,2-0,3-0,4-0,5-1,6-0 | X                                        | X                                            | X         |
| 1-0,2-0,3-0,4-0,5-1,6-1 | X                                        | X                                            | X         |
| 1-0,2-0,3-0,4-1,5-0,6-0 | X                                        | X                                            | X         |
| 1-0,2-0,3-0

In practice, full path coverage is often unachievable because the number of paths grows combinatorially with branching (path explosion), and some paths can be infeasible.